# dim_player Backfill Notebook

**Issue #166** — Rewrite to populate `dim_player` from all teams and all historical seasons
via a DB-driven, full-history approach.

Reads teams dynamically from the `team` table in the Flask app's SQLite database,
then fetches all available roster seasons per team via
`GET https://api-web.nhle.com/v1/roster-season/{team}` and upserts every player
into `dim_player` — no hardcoded team list or season constant.

## Run instructions

```bash
pip install jupyter pandas httpx sqlalchemy pytz
jupyter notebook nhl-dashboard/notebooks/dim_player_backfill.ipynb
```

Run all cells top-to-bottom.

## Notebook structure

| Section | Content |
|---|---|
| Setup | Imports, SQLAlchemy engine, NHL API base URL |
| Section 1 — Load teams from DB | Query team table for all tri-codes |
| Section 2 — Fetch roster seasons | GET /v1/roster-season/{team} per team |
| Section 3 — Upsert function | `upsert_player(session, player_dict)` |
| Section 4 — Batch upsert (all teams × all seasons) | Nested loop; commit once per team |
| Section 5 — Verification | Row count, position breakdown, sample rows |

## Setup

Imports, SQLAlchemy connection to the Flask app's SQLite database at
`instance/nhl.db`, and the NHL API base URL.
No hardcoded team list — teams are loaded from the DB in Section 1.
`DimPlayer` is imported from the backend models to keep column definitions
in sync with the live schema.

In [ ]:
import sys
import time
from datetime import datetime
from pathlib import Path

import httpx
import pandas as pd
import pytz
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# Add backend to path so we can import models without Flask app context
BACKEND_DIR = Path("../backend").resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

NHL_BASE = "https://api-web.nhle.com/v1"
DB_PATH  = BACKEND_DIR / "instance" / "nhl.db"

# Connect directly to the Flask app's SQLite DB — no app context needed
engine  = create_engine(f"sqlite:///{DB_PATH}", echo=False)
Session = sessionmaker(bind=engine)

ET = pytz.timezone("US/Eastern")


def now_eastern() -> datetime:
    """Return the current datetime in US/Eastern (naive UTC-offset stripped)."""
    return datetime.now(ET).replace(tzinfo=None)


print(f"Database  : {DB_PATH}")
print(f"DB exists : {DB_PATH.exists()}")

## Section 1 — Load teams from DB

Queries `SELECT tri_code FROM team ORDER BY tri_code` against `instance/nhl.db`
to build the `NHL_TEAMS` list dynamically. This replaces the previous hardcoded
32-team constant and picks up all 61+ tri-codes currently in the `team` table.

In [ ]:
with engine.connect() as conn:
    rows = conn.execute(text("SELECT tri_code FROM team ORDER BY tri_code")).fetchall()

NHL_TEAMS = [row[0] for row in rows]
print(f"Teams loaded from DB : {len(NHL_TEAMS)}")
print(NHL_TEAMS)

## Section 2 — Fetch roster seasons per team

Loops over `NHL_TEAMS` and calls `GET /v1/roster-season/{team}` for each team.
Stores successful results in `ROSTER_SEASONS = {tri_code: [int, ...]}` — a list of
season integers (e.g. `20242025`). Skips teams that return an HTTP error or exception.

Rate-limited at 50 ms between requests.

In [ ]:
ROSTER_SEASONS = {}

for team in NHL_TEAMS:
    try:
        r = httpx.get(f"{NHL_BASE}/roster-season/{team}", timeout=15)
        r.raise_for_status()
        seasons = r.json()
        ROSTER_SEASONS[team] = sorted(int(s) for s in seasons)
    except Exception as exc:
        print(f"  SKIP {team}: {exc}")
        continue
    time.sleep(0.05)  # polite rate-limiting — 50 ms between requests

total_combos = sum(len(s) for s in ROSTER_SEASONS.values())
print(f"Teams with season data  : {len(ROSTER_SEASONS)}")
print(f"Total team\u00d7season combos : {total_combos}")

## Section 3 — Upsert function

Defines `upsert_player(session, player_dict)` which:

1. Extracts all `dim_player` columns from the roster API player dict
2. Constructs a `DimPlayer` instance
3. Calls `session.merge()` to upsert by `player_id` (INSERT on first run, UPDATE on re-runs)
4. Sets `updated_at` to the current Eastern time

Helper `_default()` extracts `.default` from localised NHL name dicts.

In [ ]:
# Import the SQLAlchemy model — keeps column definitions in sync with live schema
from models import DimPlayer


def _default(val) -> str | None:
    """Extract .default from a localised NHL name dict, or return the value as-is."""
    return val.get("default") if isinstance(val, dict) else val


def upsert_player(session, player_dict: dict) -> None:
    """Upsert one player row into dim_player from a roster API player dict.

    Uses session.merge() so the call is idempotent: INSERT on first run,
    UPDATE on subsequent runs. sweater_number is always overwritten because
    jersey numbers change between seasons.

    Args:
        session: SQLAlchemy Session bound to the Flask app's SQLite database.
        player_dict: Raw player object from /v1/roster/{team}/{season} response.
    """
    player = DimPlayer(
        player_id        = player_dict.get("id"),
        first_name       = _default(player_dict.get("firstName")),
        last_name        = _default(player_dict.get("lastName")),
        sweater_number   = player_dict.get("sweaterNumber"),
        position         = player_dict.get("positionCode"),
        shoots_catches   = player_dict.get("shootsCatches"),
        height_in_inches = player_dict.get("heightInInches"),
        weight_in_pounds = player_dict.get("weightInPounds"),
        birth_date       = player_dict.get("birthDate"),
        birth_country    = player_dict.get("birthCountry"),
        headshot_url     = player_dict.get("headshot"),
        updated_at       = now_eastern(),
    )
    session.merge(player)


print("upsert_player() defined")
print("Columns populated:")
for col in ["player_id", "first_name", "last_name", "sweater_number", "position",
            "shoots_catches", "height_in_inches", "weight_in_pounds",
            "birth_date", "birth_country", "headshot_url", "updated_at"]:
    print(f"  {col}")

## Section 4 — Batch upsert (all teams × all seasons)

Nested loop: for each team in `NHL_TEAMS`, iterates over
`sorted(ROSTER_SEASONS.get(team, []))` in ascending order so the newest season
wins on `sweater_number`.

- Calls `GET /v1/roster/{team}/{season}`; logs and skips on error
- Upserts all players from `forwards + defensemen + goalies`
- Commits once per team (after the inner season loop) — reduces data loss if the
  outer loop fails mid-run
- Prints progress per team

In [ ]:
upserted_total = 0

for team in NHL_TEAMS:
    # ascending sort so the newest season overwrites sweater_number on the last merge
    seasons = sorted(ROSTER_SEASONS.get(team, []))
    with Session() as session:
        for season in seasons:
            try:
                r = httpx.get(f"{NHL_BASE}/roster/{team}/{season}", timeout=15)
                r.raise_for_status()
                data    = r.json()
                players = (
                    data.get("forwards",   []) +
                    data.get("defensemen", []) +
                    data.get("goalies",    [])
                )
                for p in players:
                    upsert_player(session, p)
                upserted_total += len(players)
            except Exception as exc:
                print(f"  SKIP {team}/{season}: {exc}")
                continue
            time.sleep(0.05)  # polite rate-limiting — 50 ms between requests
        session.commit()  # commit once per team, not once at the very end
    print(f"  OK {team}: {len(seasons)} season(s)")

print()
print(f"Batch complete — total players upserted: {upserted_total}")

## Section 5 — Verification

Queries `dim_player` to confirm the backfill succeeded:

1. **Total row count** — expect thousands of players across all historical seasons
2. **Position breakdown** — distribution across C, L, R, D, G
3. **Sample of 10 rows** — spot-check names, sweater numbers, and `updated_at`

In [ ]:
with engine.connect() as conn:
    # Row count
    row_count = conn.execute(text("SELECT COUNT(*) FROM dim_player")).scalar()

    # Position breakdown
    pos_rows = conn.execute(
        text("SELECT position, COUNT(*) AS cnt FROM dim_player GROUP BY position ORDER BY cnt DESC")
    ).fetchall()

    # Sample rows
    sample_rows = conn.execute(
        text(
            "SELECT player_id, first_name, last_name, sweater_number, position, "
            "birth_country, updated_at FROM dim_player LIMIT 10"
        )
    ).fetchall()

print(f"Total rows in dim_player: {row_count}")
print()
print("Position breakdown:")
for pos, cnt in pos_rows:
    print(f"  {pos or 'NULL':4s}: {cnt}")
print()

df_sample = pd.DataFrame(
    sample_rows,
    columns=["player_id", "first_name", "last_name", "sweater_number",
             "position", "birth_country", "updated_at"],
)
print(f"Sample of {len(df_sample)} rows:")
display(df_sample)